# Lab 7: Estimation & Bootstrapping — Diagnosing Broken Confidence Intervals
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 45 min Core + 15 min AI + 20 min Extension

---

**Learning Objectives:**
- Diagnose a degenerate bootstrap and distinguish it from a bootstrap that has failed silently
- Measure the true coverage of a nominal 95% interval and identify why it under-covers
- Recognise clustered data misread as i.i.d., and repair the interval with a cluster bootstrap
- Read and run the repaired procedure as an importable module (`.py`)
- Check an AI-written coverage function against coverage you measured yourself

**Dataset:** UCI Adult Census Income (32,561 respondents), plus one simulated clustered wage panel

**Format:** Every part hands you output that is *wrong*, or wrong-looking, and asks what
happened before it tells you. Write your diagnosis before running the reveal cell.

**Deliverable:** your three written diagnoses, your four answers on the AI's coverage
function, and the `src/bootstrap_utils.py` file the notebook writes.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write, in text cells or as comments:**
> - **Diagnoses 1, 2 and 3:** your diagnosis of each broken interval, written before you run the reveal: four, five and four short answers.
> - **The AI Expansion:** you revise the given prompt once, and write four short answers about the AI's coverage function, including a check against the coverage you measured in Diagnosis 3.
> - **Extension (optional):** three interpretation answers.
>
> **You run and read:** every code cell in the three diagnoses and in Ship It. They are the analyst's code, the coverage simulations, the cluster-bootstrap repair and the finished `bootstrap_utils.py` module, which is working code to learn from, not to retype. Read each cell before you run it: the questions ask about what it prints.
>
> **Code you write yourself (none of it required this week):**
> - the "Write it yourself" exercise after Ship It, about six lines in an empty cell, with a check cell;
> - the Extension's Bayesian bootstrap: two blanks, then a short comparison;
> - the take-home Challenge, a block bootstrap written from an empty cell.

In [ ]:
# GIVEN — run as-is
# Setup: libraries, seeds, and the Adult census data
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import bootstrap

Path("src").mkdir(exist_ok=True)       # Ship It writes its module into this folder

RANDOM_STATE = 42
rng = np.random.default_rng(seed=RANDOM_STATE)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ["age", "workclass", "fnlwgt", "education", "education_num",
           "marital_status", "occupation", "relationship", "race", "sex",
           "capital_gain", "capital_loss", "hours_per_week",
           "native_country", "income"]
# The file has no header row and a space after every comma
df = pd.read_csv(url, names=columns, skipinitialspace=True)

print(f"Loaded {df.shape[0]:,} rows.")

## Diagnosis 1: The Interval of Width Zero (15 min)

An analyst reports a 95% confidence interval for median age. The interval is
`[37.0, 37.0]`. They conclude the median is known exactly and move on.

Run the cell. Then write your diagnosis **before** running the reveal.

**Reading the code.** `rng.choice(ages, size=len(ages), replace=True)` draws one
bootstrap resample: as many ages as the data has, with replacement. Wrapping it in
`[... for _ in range(B)]` repeats it `B` times and collects the results in a list;
`_` is the usual name for a loop counter you never use. `.std(ddof=1)` is the
sample standard deviation, dividing by n − 1. In an f-string, `{x:.1f}` shows one
decimal, `{x:.1%}` a percent, `{x:,}` adds thousands separators, and `{x:>9}`
right-aligns in 9 characters. A number written `2_000` is 2000: the underscore is
only there to help you read it.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# The analyst's code, reproduced faithfully
ages = df["age"].values
B = 2_000

boot_medians = np.array([np.median(rng.choice(ages, size=len(ages), replace=True))
                         for _ in range(B)])

print(f"n:                 {len(ages):,}")
print(f"Sample median:     {np.median(ages):.1f}")
print(f"Bootstrap SE:      {boot_medians.std(ddof=1):.6f}")
print(f"Percentile 95% CI: [{np.percentile(boot_medians, 2.5):.1f}, "
      f"{np.percentile(boot_medians, 97.5):.1f}]")
print(f"Distinct values across {B:,} resamples: {len(np.unique(boot_medians))}")

### YOUR DIAGNOSIS

Before running the next cell, answer:

1. Is this a bug in the bootstrap, a bug in the analyst's code, or neither?
2. What property of `ages` produces this result? Name it precisely.
3. The analyst's conclusion — "the median is known exactly" — is wrong in a specific way. What is the correct reading of a bootstrap SE of zero?
4. Predict what BCa will return here, and why.

*Your diagnosis here:*

In [ ]:
# GIVEN — run as-is
# The reveal: ask scipy's BCa method for an interval on the same statistic

# catch_warnings(record=True) collects warnings in the list `caught`, so we can print them
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")          # record every warning, even a repeat
    # scipy takes a tuple of samples, so (ages,) is a tuple holding one;
    # batch=100 does the resamples 100 at a time, to keep memory small
    res = bootstrap((ages,), np.median, n_resamples=1_999, batch=100,
                    method="BCa", random_state=np.random.default_rng(RANDOM_STATE))
    for w in caught:
        print(f"WARNING RAISED: {w.category.__name__}")
        print(f"  {str(w.message)[:150]}")    # the first 150 characters of the message

print(f"\nBCa 95% CI: [{res.confidence_interval.low}, {res.confidence_interval.high}]")

**Reading the reveal.** BCa's acceleration constant comes from the jackknife: leave
one observation out and see how far the statistic moves. Here it never moves, so the
acceleration is 0/0 and scipy returns `nan` with a warning rather than a confidently
wrong number.

Read the warning, not the number. Here `nan` is the correct answer: the interval cannot
be computed from this data. A method that returns a plausible number in the same
situation is harder to catch, and Diagnosis 2 shows one.

---

## Diagnosis 2: The Interval That Lies Quietly (15 min)

No warning this time. No nan. Nominal 95% intervals, computed correctly from correct
formulas, on clean simulated data with no outliers and no missingness.

One of the four rows below is badly wrong and nothing in the output says so. A second row
is wrong in a subtler way — it disagrees with what most people expect the bootstrap to do
for them.

**What the simulation does.** Each of 400 experiments draws a sample of 50 from a
lognormal distribution, right-skewed like wages. Its true mean is
$e^{3.5 + 0.5^2/2}$ and its true median is $e^{3.5}$. Each sample gets its
intervals, and each interval is checked against the truth it is meant to cover;
coverage is the share of the 400 that do.

**Reading the code.** `cover` is a dict whose keys are pairs such as
`("mean", "bootstrap")`. `lo <= truth <= hi` is Python's chained comparison: it is
True when the truth lies inside the interval, and adding True to a count adds 1.

In [ ]:
# GIVEN — run as-is
# Coverage: run the experiment 400 times and count the hits.
# Every interval below is computed correctly from a correct formula.
true_mean = np.exp(3.5 + 0.5**2 / 2)
true_median = np.exp(3.5)
n_experiments, n_sample = 400, 50

cover = {("mean", "parametric"): 0, ("mean", "bootstrap"): 0,
         ("median", "parametric"): 0, ("median", "bootstrap"): 0}
sim_rng = np.random.default_rng(RANDOM_STATE)

for i in range(n_experiments):
    s = sim_rng.lognormal(3.5, 0.5, n_sample)

    se_i = s.std(ddof=1) / np.sqrt(n_sample)
    lo_p, hi_p = s.mean() - 1.96 * se_i, s.mean() + 1.96 * se_i

    r_mean = bootstrap((s,), np.mean, n_resamples=499, method="percentile",
                       random_state=np.random.default_rng(2_000 + i))
    r_med = bootstrap((s,), np.median, n_resamples=499, method="percentile",
                      random_state=np.random.default_rng(6_000 + i))

    cover["mean", "parametric"] += lo_p <= true_mean <= hi_p
    cover["mean", "bootstrap"] += (r_mean.confidence_interval.low <= true_mean
                                   <= r_mean.confidence_interval.high)
    cover["median", "parametric"] += lo_p <= true_median <= hi_p
    cover["median", "bootstrap"] += (r_med.confidence_interval.low <= true_median
                                     <= r_med.confidence_interval.high)

In [ ]:
# GIVEN — run as-is
# The coverage table, with the Monte Carlo standard error of each share
def mc_se(p):
    """Standard error of a share p measured over n_experiments trials."""
    return (p * (1 - p) / n_experiments) ** 0.5

print(f"True mean {true_mean:.3f} | true median {true_median:.3f} | nominal 95%\n")
print(f"{'target':>8} {'method':>12} {'coverage':>10} {'MC SE':>8}")
print("-" * 42)
for (target, label), hits in cover.items():
    p = hits / n_experiments
    print(f"{target:>8} {label:>12} {p:>9.1%} {mc_se(p):>8.1%}")

### YOUR DIAGNOSIS

1. For the **mean**, the parametric and bootstrap intervals land within Monte Carlo noise of each other, and both sit below 95%. A common claim is that the bootstrap "fixes" skewness. Explain why it does not here — what does *first-order accurate* mean, and which of the two intervals does it describe?
2. Given the MC SE column, how large would a gap have to be before you would report it as real? How many trials would you need to resolve a 2-point difference?
3. For the **median**, the parametric arm collapses. It is not a different formula — it is the same interval, evaluated against a different target. Say exactly what makes it miss, and in which direction it misses on right-skewed data.
4. The usual defence is "n = 50, that clears the n > 30 rule." State precisely what that rule does and does not guarantee.
5. A firm runs 1,000 experiments a year against a 95% nominal standard, reporting medians. Convert the coverage gap you measured into extra wrong calls per year, and state the assumption your conversion needs.

*Your diagnosis here:*

---

## Diagnosis 3: The Interval That Is Far Too Narrow (15 min)

The most expensive of the three, and the one that appears most often in applied
microeconomics. The data below are wages for 1,000 workers across 25 firms. Workers at
the same firm resemble one another — same pay bands, same local labour market.

The analyst bootstraps them as 1,000 independent draws.

**Reading the code.** `firm_effect[:, None]` turns the 25 firm effects into a column,
so NumPy adds each firm's effect to all 40 of its workers. `.ravel()` flattens the
25 × 40 array of wages into one row of 1,000, and `np.repeat(np.arange(25), 40)`
gives the matching firm labels: forty 0s, then forty 1s, and so on.

In [ ]:
# GIVEN — run as-is
# Build a clustered panel with a known true mean
N_FIRMS, PER_FIRM = 25, 40
TRUE_MEAN = 60_000.0

clus_rng = np.random.default_rng(RANDOM_STATE)
firm_effect = clus_rng.normal(0, 9_000, N_FIRMS)     # firms differ a lot
worker_noise = clus_rng.normal(0, 3_000, (N_FIRMS, PER_FIRM))

wages = TRUE_MEAN + firm_effect[:, None] + worker_noise
panel = pd.DataFrame({
    "firm": np.repeat(np.arange(N_FIRMS), PER_FIRM),
    "wage": wages.ravel(),
})

print(f"{len(panel):,} workers across {panel['firm'].nunique()} firms")
# the SD of the 25 firm means, then the average SD of wages inside one firm
print(f"Between-firm SD: {panel.groupby('firm')['wage'].mean().std(ddof=1):>8,.0f}")
print(f"Within-firm SD:  {panel.groupby('firm')['wage'].std(ddof=1).mean():>8,.0f}")
print(f"\nSample mean wage: ${panel['wage'].mean():>9,.0f}   (true mean ${TRUE_MEAN:,.0f})")

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# The analyst's interval: resample WORKERS
w = panel["wage"].values
naive = np.array([clus_rng.choice(w, size=len(w), replace=True).mean()
                  for _ in range(2_000)])

lo_n, hi_n = np.percentile(naive, [2.5, 97.5])
print(f"Naive (i.i.d.) bootstrap 95% CI: [${lo_n:,.0f}, ${hi_n:,.0f}]")
print(f"  width ${hi_n - lo_n:,.0f}")
print(f"  contains the true mean? {lo_n <= TRUE_MEAN <= hi_n}")

### YOUR DIAGNOSIS

1. The naive interval may well contain the true mean *this time*. Explain why that does not make it correct.
2. Which of Chapter 7's five bootstrap failure modes is this, and what exactly does resampling workers destroy?
3. The effective sample size here is not 1,000. Argue for a better number and say what it depends on.
4. Write the fix in one sentence before you look at the next cell.

*Your diagnosis here:*

In [ ]:
# GIVEN — run as-is
# The repair: resample FIRMS, keeping each firm intact
# One array of wages per firm; the loop does not need the firm label, hence _
groups = [g["wage"].values for _, g in panel.groupby("firm")]

def cluster_boot_mean(groups, B, rng):
    """Resample whole clusters with replacement, then pool."""
    k = len(groups)
    out = np.empty(B)                          # room for B results, filled below
    for b in range(B):
        picked = rng.integers(0, k, size=k)    # k firm numbers, drawn with replacement
        out[b] = np.concatenate([groups[i] for i in picked]).mean()
    return out

clustered = cluster_boot_mean(groups, 2_000, np.random.default_rng(RANDOM_STATE))
lo_c, hi_c = np.percentile(clustered, [2.5, 97.5])
ratio_d3 = (hi_c - lo_c) / (hi_n - lo_n)

print(f"Naive     95% CI: [${lo_n:,.0f}, ${hi_n:,.0f}]   width ${hi_n - lo_n:,.0f}")
print(f"Clustered 95% CI: [${lo_c:,.0f}, ${hi_c:,.0f}]   width ${hi_c - lo_c:,.0f}")
print(f"\nWidth ratio, clustered / naive: {ratio_d3:.1f}x")

**Verification checkpoint:** the clustered interval should be roughly 4-6x wider than
the naive one. (Measured: 5.7x.)

The naive interval is too narrow for this data, so a test built on it would reject a
true null far more often than 5% of the time.

A wider interval is not necessarily a correct one, so the next two cells measure
coverage. The first puts Diagnosis 3's design and its two intervals into functions of a
trial number `t`; the second draws 200 fresh panels and counts how often each interval
contains the true mean.

In [ ]:
# GIVEN — run as-is
# Diagnosis 3's design and its two intervals, as functions of the trial number t
def one_panel(t):
    """Trial t's panel, from seed 5_000 + t: a list of 25 arrays of wages."""
    panel_rng = np.random.default_rng(5_000 + t)
    firm_effect = panel_rng.normal(0, 9_000, N_FIRMS)
    worker_noise = panel_rng.normal(0, 3_000, (N_FIRMS, PER_FIRM))
    return [TRUE_MEAN + firm_effect[i] + worker_noise[i] for i in range(N_FIRMS)]

def naive_ci(groups, t):
    """The analyst's 95% interval: resample workers 299 times, from seed 9_000 + t."""
    flat = np.concatenate(groups)
    boot_rng = np.random.default_rng(9_000 + t)
    draws = [boot_rng.choice(flat, size=len(flat), replace=True).mean() for _ in range(299)]
    return np.percentile(draws, [2.5, 97.5])

def cluster_ci(groups, t):
    """The repaired 95% interval: resample firms 299 times, from seed 11_000 + t."""
    draws = cluster_boot_mean(groups, 299, np.random.default_rng(11_000 + t))
    return np.percentile(draws, [2.5, 97.5])

In [ ]:
# GIVEN — run as-is
# Coverage: how often each interval contains the true mean, over 200 panels
TRIALS = 200
hit_naive = 0
hit_clust = 0
for t in range(TRIALS):
    gs = one_panel(t)
    lo, hi = naive_ci(gs, t)
    hit_naive += lo <= TRUE_MEAN <= hi
    lo, hi = cluster_ci(gs, t)
    hit_clust += lo <= TRUE_MEAN <= hi

print(f"Naive     bootstrap coverage over {TRIALS} panels: {hit_naive / TRIALS:.1%}")
print(f"Clustered bootstrap coverage over {TRIALS} panels: {hit_clust / TRIALS:.1%}")

Nominal is 95%; the naive interval covered 19.5%, and nothing in its output said so.

---

## Ship It: `src/bootstrap_utils.py`

The repaired procedure goes into a module, so the next analysis imports it instead of
re-deriving it, and the cluster argument is a parameter rather than something someone has
to remember.

Here is the finished module. Read `cluster_bootstrap_mean` against `cluster_boot_mean`
in the repair above, then run the two cells: the first writes the file, the second
imports it and re-runs the clustered interval through it.

In [ ]:
%%writefile src/bootstrap_utils.py
"""Bootstrap helpers for clustered and skewed economic data (ECON 5200 Lab 7).

The cluster argument is a parameter, not an afterthought: pass the clusters to
cluster_bootstrap_mean and the interval respects the dependence inside each
cluster; bootstrap_mean treats every row as an independent draw.
"""
import numpy as np


def percentile_ci(draws, level=0.95):
    """Percentile interval from an array of bootstrap replicates."""
    alpha = (1.0 - level) / 2.0
    lo, hi = np.percentile(draws, [100 * alpha, 100 * (1 - alpha)])
    return float(lo), float(hi)


def bootstrap_mean(x, B=2000, seed=42):
    """Ordinary i.i.d. bootstrap of the mean. Valid only for independent draws."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    return np.array([rng.choice(x, size=len(x), replace=True).mean()
                     for _ in range(B)])


def cluster_bootstrap_mean(groups, B=2000, seed=42):
    """Cluster bootstrap of the mean: resample whole clusters, then pool.

    `groups` holds one array per cluster. With few clusters the interval is much
    wider than the i.i.d. one; that width is the honest cost of the design.
    For regression with few clusters, prefer the wild cluster bootstrap
    (Cameron, Gelbach & Miller 2008).
    """
    rng = np.random.default_rng(seed)
    groups = [np.asarray(g) for g in groups]
    k = len(groups)
    if k < 2:
        raise ValueError("need at least 2 clusters")
    out = np.empty(B)
    for b in range(B):
        picked = rng.integers(0, k, size=k)
        out[b] = np.concatenate([groups[i] for i in picked]).mean()
    return out

In [ ]:
# GIVEN — run as-is
# Import the module and re-run Diagnosis 3's repair through it
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import bootstrap_utils as bu
importlib.reload(bu)               # reload, so a re-run of the %%writefile cell takes effect

check = bu.cluster_bootstrap_mean(groups, B=500, seed=RANDOM_STATE)
print(f"Imported and re-ran: 95% CI {bu.percentile_ci(check)}")

---

### Write it yourself (optional this week)

This exercise is optional and ungraded this week. It is the one place in the core of
this lab where you write code from an empty cell.

Diagnosis 3's firms differ far more than the workers inside them: a firm SD of 9,000
against a worker SD of 3,000. Turn that round. In the empty cell below, build one
panel of the same size, 25 firms of 40 workers with a true mean of 60,000, but with a
firm SD of **3,000** and a worker SD of **9,000**. `one_panel` shows how; any seed
will do. Then give it both of Diagnosis 3's intervals, `naive_ci` and `cluster_ci` (their
second argument is a trial number; any will do), and set `width_ratio` to the clustered width divided by the naive width.

It takes about six lines. Then run the check cell. If you skipped the exercise it
says so and moves on; if you wrote it, it fails until your ratio is in the right
range. Compare your ratio with Diagnosis 3's, and ask what made the difference.

In [ ]:
# WRITE IT YOURSELF — optional: width_ratio for the flipped panel

In [ ]:
# CHECK — for the optional exercise above
# globals() holds every name defined so far in this notebook
if "width_ratio" not in globals():
    print("Optional exercise not attempted: nothing to check. That is fine this week.")
else:
    assert width_ratio < 4, (
        f"width_ratio is {width_ratio:.1f}, close to Diagnosis 3's {ratio_d3:.1f}. Did you "
        "build a new panel with a firm SD of 3,000 and a worker SD of 9,000?")
    assert width_ratio > 0.75, (
        f"width_ratio is {width_ratio:.2f}, well below 1. Divide the clustered width "
        "by the naive width.")
    assert width_ratio > 1.2, (
        f"width_ratio is {width_ratio:.2f}: both intervals behave as if the workers were "
        "independent. Is one of them resampling firms?")
    print(f"Flipped panel: clustered / naive width = {width_ratio:.1f}x "
          f"(Diagnosis 3: {ratio_d3:.1f}x)")

---
## AI-Assisted Expansion: a coverage function written by AI, checked by you

Diagnosis 3 measured coverage with a loop written for that one experiment. The same
loop, made general, checks any interval on any simulated data: draw a dataset, build
the interval, count a hit when it contains the truth, repeat. That is routine code
and easy to get subtly wrong, so it is a fair job for an AI, as long as you check the
result. Ask an AI to write it, run it on Diagnosis 3's two intervals, and compare it
with the coverage you already measured.

**What to do**
1. Copy **prompt version 1** below into Claude ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT.
2. Read what comes back before you run it. The prompt leaves at least one
   important thing unsaid, so the AI had to guess. Find one guess that is wrong,
   or one that you would rather decide yourself. Pasting the version 1 function
   into the empty code cell and running the check is one way to find it.
3. Write **prompt version 2** in the cell below, fixing that one thing, and ask
   again. Paste the function from version 2 into the empty code cell and run the
   check.
4. Answer the questions at the end. They are what is graded, not the code.

**Reading the check.** The check runs the AI's function on Diagnosis 3's two intervals
and prints its coverage beside yours. An exact match needs repetition `t` to use the same
seeds Diagnosis 3 used. Any other seeding is still a correct simulation, but it draws
different panels, and two estimates from 200 trials each differ by chance: the check
prints the standard error of that gap. Gaps of up to about two of them (8 points for
the naive interval) are common, and a little more happens now and then: that is noise,
not an error. A share above 100%, or a crash, is an error.

In [ ]:
# PROMPT, version 1 — copy it exactly as it is
#
# Write a Python function coverage(interval_fn, draw_sample, true_value, n_reps, seed)
# that measures how often a confidence interval contains the true value.
# draw_sample(seed) returns one simulated dataset, and interval_fn(data, seed)
# returns a (low, high) interval for it. Repeat this n_reps times and return
# the share of intervals that contain true_value. Use NumPy only, and comment
# each line.

# PROMPT, version 2 — your revision. Keep everything from version 1 that was
# fine, and add or change the part the AI had to guess.
#
#

In [ ]:
# Paste the coverage function from your version 2 here, then run the check below.

In [ ]:
# CHECK — the AI's coverage function against Diagnosis 3's coverage
# The check passes seed=0. It reproduces Diagnosis 3 only if repetition t gives
# the seed t to both draw_sample and interval_fn.
if "coverage" not in globals():
    print("Paste the AI's coverage function in the cell above first.")
else:
    # try runs the block; if it raises a TypeError, the except block runs instead
    try:
        for name, interval_fn, hits in [("naive", naive_ci, hit_naive),
                                        ("clustered", cluster_ci, hit_clust)]:
            ours = hits / TRIALS
            gap_se = (2 * ours * (1 - ours) / TRIALS) ** 0.5    # SE of the gap between two estimates
            ai = coverage(interval_fn, one_panel, TRUE_MEAN, TRIALS, 0)
            print(f"{name:>9}: AI {ai:.1%}   Diagnosis 3 {ours:.1%}   (SE of the gap {gap_se:.1%})")
        print("If a pair differs, find out why before you trust the function.")
    except TypeError as err:
        print(f"coverage passed something other than a whole number as the seed. ({err})")

### Document the interaction

This is what is graded for the expansion.

1. **Your revision.** What did you change between version 1 and version 2, and
   what did the AI get wrong or have to guess that made you change it?
2. **The check.** Do the AI's two coverages match Diagnosis 3's exactly? If a pair
   differs, how many SEs of the gap apart are they, and is that noise or an error?
   What in the AI's function caused the difference? Would a gap that size change
   your diagnosis of the naive interval?
3. **In your own words.** Pick the line or two of the AI's function that decide
   which seed each repetition uses and whether an interval counts as a hit, and
   explain in 2–3 sentences what they do.
4. **Would you use AI for this again?** One sentence: when is it worth it, and
   when is it faster to write the function yourself?

*Your answers here:*

1.
2.
3.
4.

---

## Extension (Optional — 20 min): The Bayesian Bootstrap

Rubin (1981) replaces resampling with reweighting: instead of drawing observations with
probability 1/n, draw a weight vector from a Dirichlet(1, ..., 1) and take a weighted mean.
The result approximates the posterior under a non-informative prior — Bayesian inference with
no MCMC, and trivially parallel.

In [ ]:
# YOUR TASK — Implement the Bayesian bootstrap and compare
cap_gain = df.loc[df["capital_gain"] > 0, "capital_gain"].values
sub = cap_gain[:500]                       # the first 500, to keep it quick


def bayesian_bootstrap_mean(x, B, seed):
    """Dirichlet(1,...,1) weights instead of resampling."""
    r = np.random.default_rng(seed)
    x = np.asarray(x)
    # YOUR CODE: draw B weight vectors from Dirichlet(1,...,1), each of length len(x),
    # and return the B weighted means. The Generator r has a dirichlet method.
    weights = ____
    return ____

# Compare the two intervals on the same data.
# They should agree closely at n = 500 — the interesting question is why,
# and where they would come apart.

### Interpretation Questions

1. The frequentist and Bayesian intervals nearly coincide at n = 500. State the asymptotic result that guarantees this, and name a sample size where you would expect visible disagreement.
2. The two intervals support different sentences. Write the one sentence each licenses, and mark the word that cannot be moved between them.
3. Chapter 7's Tier B block gives the wild cluster bootstrap for regression with few clusters. Diagnosis 3 used a plain cluster bootstrap for a mean. Explain what the Rademacher weights buy that resampling clusters does not, and why it matters most at 10 clusters rather than 25.

*Your answers here:*

---

## Challenge (Take-Home): The Fourth Failure Mode

Diagnosis 3 covered dependence across firms. Dependence across *time* breaks the bootstrap
in the same way and is repaired differently — resample contiguous **blocks** rather than
individual periods, so that the local autocorrelation survives.

This is the preview of Chapter 20, where the same dependence stops being a nuisance
and becomes the object of study.

In [ ]:
# YOUR TASK — Block bootstrap on a persistent series

# Step 1: simulate an AR(1) with phi = 0.8 and 500 periods.

# Step 2: bootstrap the series mean two ways —
#   (a) i.i.d. resampling of periods
#   (b) moving-block bootstrap with block length 20

# Step 3: report both interval widths. Predict the ordering first.

# Step 4: block length is a bias-variance tradeoff. Argue for a rule
#   relating block length to phi, and say what goes wrong at both extremes.

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed a degenerate bootstrap: the median of 32,561 integer ages
  had a bootstrap SE of [YOUR VALUE], and BCa returned nan with a warning
  instead of an interval
* Measured the coverage of nominal 95% intervals on skewed simulated
  data (n = 50, 400 experiments): for the mean, parametric [YOUR VALUE]%
  and bootstrap [YOUR VALUE]%; for the median, parametric [YOUR VALUE]%
  and bootstrap [YOUR VALUE]%
* Showed that wages from 25 firms bootstrapped as independent workers
  gave an interval [YOUR VALUE]x too narrow, with [YOUR VALUE]% coverage;
  resampling whole firms brought coverage to [YOUR VALUE]%
* Used a bootstrap_utils.py module with percentile, i.i.d. and cluster
  bootstrap functions
* Had an AI write a coverage function, revised my prompt once, and
  checked it against my own coverage numbers: [YOUR VALUE]

**Please write a README.md entry including:**
1. Project Title: Diagnosing Broken Confidence Intervals
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust',
'comprehensive' or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab07-bootstrapping`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab07-bootstrapping`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab07-bootstrapping`, branch `main`,
   commit message `Lab 07 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. `bootstrap_utils.py` is **not** pushed by step 2 — that step sends the notebook
   only, and the file lives in Colab's temporary storage, which is wiped when the
   session ends. Download it first: in the Colab **Files** panel (folder icon,
   left), open `src`, right-click `bootstrap_utils.py` → **Download**. Then on the
   repo page use **Add file → Upload files**, drag it in, and **Commit changes**.
   It is part of what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.